### N-th Primes and Prime Less Than N

This notebook contains several method to calculate `n`-th prime number and all primes less than a number `n`. All functions have no input-guard so please use with attention.

Hope you enjoy these code! If you find any bug or have any question, don't hesitate to contact me!

First let's have a short prelude.

In [ ]:
import time
import math


def bench_prime_less_than(func, n):
    start = time.perf_counter()
    prime_list = func(int(n))
    end = time.perf_counter()
    print(f"num of primes less than {n}: {len(prime_list)}, time consumed: {end-start}")


def wrapper_nth_prime(func, n: int):
    primes = []
    trial = 2
    while len(primes) < n:
        if func(trial, primes):
            primes.append(trial)
        trial += 1
    return primes[-1]


def bench_check_prime_until(func, n):
    start = time.perf_counter()
    res = wrapper_nth_prime(func, n)
    end = time.perf_counter()
    print(f"{n}-th prime is: {res}, time consumed: {end-start}")

Below are functions for `n`-th prime. 

- `check_prime` uses naive method
- `check_prime_2` use a slightly cleverer method.
- `check_prime_3` use calculated prime numbers to accelerate calculation, but will depend on outside modifiable `prime_list` to keep track of primes.

In [ ]:
def check_prime(n: int, prime_list: list):
    for i in range(2, n + 1):
        trail_num = n / i
        is_integer = int(trail_num) == trail_num
        if is_integer and trail_num != 1:
            return False
    return True


def check_prime_2(n: int, prime_list: list):
    for i in range(2, math.isqrt(n) + 1):
        trail_num = n / i
        is_integer = int(trail_num) == trail_num
        if is_integer and trail_num != 1:
            return False
    return True


def check_prime_3(n: int, prime_list: list):
    if n == 2:
        return True
    boundary = math.isqrt(n)
    for p in prime_list:
        if p > boundary:
            break

        trail_num = n / p
        is_integer = int(trail_num) == trail_num
        if is_integer and trail_num != 1:
            return False
    return True

And a small benchmark:

In [ ]:
bench_check_prime_until(check_prime, 1000)
bench_check_prime_until(check_prime_2, 1000)
bench_check_prime_until(check_prime_3, 1000)

So you can see it's indeed improving. But these method still can be improved, especially when they meet Eratosthenes sieve method.

In [ ]:
import math
from time import perf_counter as pc

def seive_method(n: int):
    sieve = [True] * (n + 1)
    sieve[0] = sieve[1] = False
    for p in range(2, math.isqrt(n) + 1):
        if sieve[p] == True:
            for j in range(p * p, n + 1, p):
                sieve[j] = False
    prime_list = [i for i, is_prime in enumerate(sieve) if is_prime]
    return prime_list


def by_definition_method(n: int):
    p_list = []
    is_prime = True
    for i in range(2, n + 1):
        if i == 2:
            is_prime = True
        boundary = math.isqrt(i)
        for p in p_list:
            if p > boundary:
                is_prime = True
                break

            trail_num = i / p
            is_integer = int(trail_num) == trail_num
            if is_integer and trail_num != 1:
                is_prime = False
                break
        p_list.append(i) if is_prime == True else 1
        is_prime = True

    return p_list


big_n = int(1e6)

start_seive = pc()
prime_list_seive = seive_method(big_n)
end_seive = pc()
print(f"Seive method with n = {big_n} time consuming: {end_seive - start_seive}")

start_naive = pc()
prime_list_naive = by_definition_method(big_n)
end_naive = pc()
print(f"Naive method with n = {big_n} time consuming: {end_naive - start_naive}")

assert prime_list_seive == prime_list_naive

Below are several implementations of classical sieve method and different improvement. Notice that two functions, `prime_lt_seg_byte_parallel` and `prime_lt_seg_byte_2wheel_parallel`, should be imported from other files.

If you wish, you could also use `prime_lt` and `prime_lt_byte` defined in module `multi_process_prime`.

In [ ]:
def prime_lt(n: int):
    sieve = [True] * (n + 1)
    sieve[0] = sieve[1] = False
    for p in range(2, math.isqrt(n) + 1):
        if sieve[p] == True:
            for j in range(p * p, n + 1, p):
                sieve[j] = False
    prime_list = [i for i, is_prime in enumerate(sieve) if is_prime]
    return prime_list


def prime_lt_byte(n: int):
    if n < 2:
        return []

    sieve = bytearray(b"\x01") * (n + 1)
    sieve[:2] = b"\x00\x00"

    for p in range(2, math.isqrt(n) + 1):
        if sieve[p]:
            start = p * p
            sieve[start : n + 1 : p] = b"\x00" * ((n - start) // p + 1)

    return [i for i in range(2, n + 1) if sieve[i]]


def prime_lt_byte_2wheel(n: int):
    if n < 2:
        return []

    size = (n + 1) // 2
    sieve = bytearray(b"\x01") * size
    sieve[:1] = b"\x00"

    for i in range(1, math.isqrt(n) // 2 + 1):
        if sieve[i]:
            p = 2 * i + 1
            start = 2 * i * i + 2 * i
            count = (size - 1 - start) // p + 1
            sieve[start:size:p] = b"\x00" * count

    return [2] + [2 * i + 1 for i in range(1, size) if sieve[i]]


def prime_lt_seg(n: int):

    if n < 2:
        return []

    seg_len = math.isqrt(n)
    base_primes = prime_lt(seg_len)
    prime_list = base_primes.copy()

    low = seg_len
    while low <= n:
        high = min(low + seg_len - 1, n)
        sieve = [True] * (high - low + 1)

        for p in base_primes:
            start = max(p * p, ((low + p - 1) // p) * p)
            for j in range(start, high + 1, p):
                sieve[j - low] = False

        prime_list.extend(
            [prime + low for prime, is_prime in enumerate(sieve) if is_prime]
        )
        low += seg_len

    return prime_list


def prime_lt_seg_range(n: int):

    if n < 2:
        return []

    seg_len = math.isqrt(n)
    base_primes = prime_lt(seg_len)
    prime_list = base_primes.copy()

    low = seg_len
    while low <= n:
        high = min(low + seg_len - 1, n)
        sieve = [True] * (high - low + 1)

        for p in base_primes:
            start = max(p * p, ((low + p - 1) // p) * p)
            count = (high - start) // p + 1
            sieve[start - low : high - low + 1 : p] = [False] * count

        prime_list.extend(
            [prime + low for prime, is_prime in enumerate(sieve) if is_prime]
        )
        low += seg_len

    return prime_list


def prime_lt_seg_range_byte(n: int):

    if n < 2:
        return []

    seg_len = math.isqrt(n)
    base_primes = prime_lt(seg_len)
    prime_list = base_primes.copy()

    low = seg_len
    while low <= n:
        high = min(low + seg_len - 1, n)
        sieve = bytearray(b"\x01") * (high - low + 1)

        for p in base_primes:
            start = max(p * p, ((low + p - 1) // p) * p)
            count = (high - start) // p + 1
            sieve[start - low : high - low + 1 : p] = b"\x00" * count

        prime_list.extend(
            [prime + low for prime, is_prime in enumerate(sieve) if is_prime]
        )
        low += seg_len

    return prime_list


from multi_process_prime import prime_lt_seg_byte_parallel, prime_lt_seg_byte_2wheel_parallel

Of course, some more benchmark:

In [ ]:
big_n = int(10000)
bench_prime_less_than(prime_lt, big_n)
bench_prime_less_than(prime_lt_byte_2wheel, big_n)
bench_prime_less_than(prime_lt_byte, big_n)
bench_prime_less_than(prime_lt_seg, big_n)
bench_prime_less_than(prime_lt_seg_range, big_n)
bench_prime_less_than(prime_lt_seg_range_byte, big_n)
bench_prime_less_than(prime_lt_seg_byte_parallel, big_n)
bench_prime_less_than(prime_lt_seg_byte_2wheel_parallel, big_n)

Hope you enjoy!